In [ ]:
# Install + imports
# This notebook is intentionally kept close to the original repo logic,
# while being fully self-contained and runnable in Colab/Jupyter.

!python -V

!pip install -q timm torchvision ultralytics pycocotools matplotlib seaborn pandas pillow einops torchmetrics

import os
import random
import csv
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from einops import rearrange
from torchmetrics.detection import MeanAveragePrecision

import timm
from ultralytics.nn.modules.head import Detect
from ultralytics.utils.loss import v8DetectionLoss

print('Torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Timm', timm.__version__)
print('Ultralytics OK')


In [ ]:
# Project config and options, aligned closely with the repo
ROOT = Path('/content') if os.path.exists('/content') else Path.cwd()
DATA_ROOT = ROOT / 'data'
FOCI_RAW = DATA_ROOT / 'foci_raw'
FOCI_PREP = DATA_ROOT / 'foci'
CHECKPOINT_DIR = ROOT / 'checkpoints'
LOG_DIR = ROOT / 'logs'
FIG_DIR = ROOT / 'figures'
RESULTS_DIR = ROOT / 'results'

for d in [DATA_ROOT, FOCI_PREP, CHECKPOINT_DIR, LOG_DIR, FIG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED_VALUES = [42, 43, 44]
DEFAULT_SEED = 42

random.seed(DEFAULT_SEED)
np.random.seed(DEFAULT_SEED)
torch.manual_seed(DEFAULT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(DEFAULT_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Repo-like defaults
CLASS_NAMES = ['virus']
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 640
BATCH_SIZE = 4
EPOCHS = 60
LR_BACKBONE = 1e-4
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.05
DROP_PATH = 0.1
CLOSE_MOSAIC = 10
EVAL_EVERY = 5
USE_P2 = False

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

AUG_PRESETS = {
    'A': {'mosaic': 0.0, 'mixup': 0.0, 'erase': 0.0, 'photo_ops': 0},
    'B': {'mosaic': 1.0, 'mixup': 0.0, 'erase': 0.0, 'photo_ops': 0},
    'C': {'mosaic': 1.0, 'mixup': 0.0, 'erase': 0.25, 'photo_ops': 2},
    'D': {'mosaic': 1.0, 'mixup': 0.1, 'erase': 0.25, 'photo_ops': 2},
}

# Default run is just aug A, but you can add more presets here if needed.
# Example: AUG_OPTIONS = ['A', 'B']
AUG_OPTIONS = ['A']

os.environ['PYTHONHASHSEED'] = str(DEFAULT_SEED)


In [ ]:
# FOCI dataset preparation and split creation, matching the repo project flow
from xml.etree import ElementTree as ET
import shutil
import random as py_random


def find_raw_foci_dir():
    candidates = [
        DATA_ROOT / 'foci_raw',
        DATA_ROOT / 'foci',
        Path('/kaggle/input'),
        Path('/mnt/data'),
    ]
    for c in candidates:
        if c.is_dir():
            image_dir = c / 'JPEGImages' / 'JPEGImages'
            ann_dir = c / 'Virus_anno' / 'Annotations'
            if image_dir.exists() and ann_dir.exists():
                return c
    return DATA_ROOT / 'foci_raw'


def prepare_foci_dataset(force=False, val_size=14, seed=42):
    global FOCI_RAW, FOCI_PREP
    FOCI_RAW = find_raw_foci_dir()
    FOCI_PREP = DATA_ROOT / 'foci'

    if force and FOCI_PREP.exists():
        shutil.rmtree(FOCI_PREP)

    image_dir = FOCI_RAW / 'JPEGImages' / 'JPEGImages'
    ann_dir = FOCI_RAW / 'Virus_anno' / 'Annotations'

    if not image_dir.exists() or not ann_dir.exists():
        raise FileNotFoundError(
            f'FOCI raw dataset not found. Expected: {image_dir} and {ann_dir}. '\
            'Place dataset under /content/data/foci_raw or /kaggle/input/...'
        )

    if not FOCI_PREP.exists() or not (FOCI_PREP / 'images' / 'train').exists() or force:
        print(f'Preparing FOCI dataset from {FOCI_RAW}...')
        images = sorted([p for p in image_dir.rglob('*') if p.suffix.lower() in IMG_EXTS])
        items = []
        missing = []

        for img in images:
            exact = ann_dir / f'{img.stem}.xml'
            if exact.exists():
                items.append((img, exact))
            else:
                matches = list(ann_dir.rglob(f'{img.stem}.xml'))
                if len(matches) == 1:
                    items.append((img, matches[0]))
                elif len(matches) == 0:
                    missing.append(img.name)
                else:
                    raise ValueError(f'Multiple XML files for {img.name}: {matches}')

        if missing:
            print(f'Skipped {len(missing)} images without annotations')

        if not items:
            raise ValueError('No annotated images found in FOCI dataset.')

        py_random.Random(seed).shuffle(items)
        val_items = items[:val_size]
        train_items = items[val_size:]

        out_img = FOCI_PREP / 'images'
        out_lab = FOCI_PREP / 'labels'
        for split_name, split_items in [('train', train_items), ('val', val_items)]:
            (out_img / split_name).mkdir(parents=True, exist_ok=True)
            (out_lab / split_name).mkdir(parents=True, exist_ok=True)
            for img, xml_path in split_items:
                dst_img = out_img / split_name / img.name
                if not dst_img.exists():
                    shutil.copy2(str(img), str(dst_img))
                root = ET.parse(str(xml_path)).getroot()
                with open(out_lab / split_name / f'{img.stem}.txt', 'w') as f:
                    for obj in root.iter('object'):
                        name = obj.findtext('name', '').strip().lower()
                        if name != 'virus':
                            continue
                        box = obj.find('bndbox')
                        xmin = float(box.findtext('xmin'))
                        ymin = float(box.findtext('ymin'))
                        xmax = float(box.findtext('xmax'))
                        ymax = float(box.findtext('ymax'))
                        cx = (xmin + xmax) / 2.0 / 1024.0
                        cy = (ymin + ymax) / 2.0 / 768.0
                        w = (xmax - xmin) / 1024.0
                        h = (ymax - ymin) / 768.0
                        f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

        print(f'Prepared dataset: {FOCI_PREP}')
        print(f'train={len(train_items)} val={len(val_items)}')
    else:
        print(f'Using existing FOCI dataset at {FOCI_PREP}')


def load_foci_split(split_name='train'):
    image_dir = FOCI_PREP / 'images' / split_name
    label_dir = FOCI_PREP / 'labels' / split_name
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(f'Missing split folder: {image_dir} / {label_dir}')

    items = []
    for p in sorted(image_dir.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            label_p = label_dir / f'{p.stem}.txt'
            if label_p.exists():
                items.append((p, label_p))
    return items


prepare_foci_dataset(force=False, val_size=14, seed=42)
train_items = load_foci_split('train')
val_items = load_foci_split('val')
print('Train items:', len(train_items))
print('Val items:', len(val_items))

for img, lbl in train_items[:2]:
    print(f'{img.name}:')
    print(lbl.read_text()[:200])


In [ ]:
# ViTBase backbone + feature pyramid + YOLO head
# This is the repo's architecture, reproduced as a notebook.


def window_partition(x, ws):
    return rearrange(x, 'b (nh ws1) (nw ws2) c -> (b nh nw) (ws1 ws2) c', ws1=ws, ws2=ws)


def window_unpartition(x, ws, H, W):
    return rearrange(x, '(b nh nw) (ws1 ws2) c -> b (nh ws1) (nw ws2) c', nh=H // ws, nw=W // ws, ws1=ws, ws2=ws)


class SimpleFeaturePyramid(nn.Module):
    def __init__(self, in_ch=384, out_ch=128, use_p2=False):
        super().__init__()
        self.use_p2 = use_p2
        self.strides = (4, 8, 16, 32) if use_p2 else (8, 16, 32)

        def head(c):
            return nn.Sequential(
                nn.Conv2d(c, out_ch, 1, bias=False),
                nn.GroupNorm(8, out_ch),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.GroupNorm(8, out_ch),
            )

        if use_p2:
            self.to_p2 = nn.Sequential(
                nn.ConvTranspose2d(in_ch, in_ch // 2, 2, 2),
                nn.GroupNorm(8, in_ch // 2),
                nn.GELU(),
                nn.ConvTranspose2d(in_ch // 2, in_ch // 4, 2, 2),
            )
            self.head_p2 = head(in_ch // 4)

        self.to_p3 = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, 2)
        self.to_p4 = nn.Identity()
        self.to_p5 = nn.MaxPool2d(2, 2)
        self.head_p3 = head(in_ch // 2)
        self.head_p4 = head(in_ch)
        self.head_p5 = head(in_ch)

    def forward(self, x):
        outs = [self.head_p3(self.to_p3(x)), self.head_p4(self.to_p4(x)), self.head_p5(self.to_p5(x))]
        if self.use_p2:
            outs.insert(0, self.head_p2(self.to_p2(x)))
        return outs


class ViTDetBackbone(nn.Module):
    def __init__(self, img_size=640, window_size=10, global_idx=(2, 5, 8, 11), drop_path_rate=0.0):
        super().__init__()
        self.vit = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0,
                                     img_size=img_size, drop_path_rate=drop_path_rate)
        self.patch_size = 16
        self.embed_dim = self.vit.embed_dim
        self.window_size = window_size
        self.global_idx = set(global_idx)

        g = img_size // self.patch_size
        assert g % window_size == 0, f'grid {g} must be divisible by window_size {window_size}'

        pos = self.vit.pos_embed[:, 1:, :].detach().clone()
        self.pos_embed = nn.Parameter(rearrange(pos, '1 (h w) c -> 1 h w c', h=g, w=g))

    @staticmethod
    def run_attention(attn, x):
        q, k, v = rearrange(attn.qkv(x), 'b n (three h d) -> three b h n d', three=3, h=attn.num_heads)
        out = F.scaled_dot_product_attention(q, k, v)
        return attn.proj(rearrange(out, 'b h n d -> b n (h d)'))

    def forward(self, x):
        x = rearrange(self.vit.patch_embed.proj(x), 'b c h w -> b h w c')
        x = x + self.pos_embed
        B, H, W, C = x.shape

        for i, blk in enumerate(self.vit.blocks):
            y = blk.norm1(x)
            if i in self.global_idx:
                y = rearrange(y, 'b h w c -> b (h w) c')
                y = rearrange(self.run_attention(blk.attn, y), 'b (h w) c -> b h w c', h=H, w=W)
            else:
                y = window_unpartition(self.run_attention(blk.attn, window_partition(y, self.window_size)),
                                       self.window_size, H, W)
            x = x + blk.drop_path1(blk.ls1(y))
            x = x + blk.drop_path2(blk.ls2(blk.mlp(blk.norm2(x))))

        x = self.vit.norm(x)
        return rearrange(x, 'b h w c -> b c h w')


class YoloHead(nn.Module):
    def __init__(self, num_classes=1, ch=(128, 128, 128), strides=(8, 16, 32)):
        super().__init__()
        self.detect = Detect(nc=num_classes, ch=ch)
        self.detect.stride = torch.tensor(strides, dtype=torch.float32)
        self.detect.bias_init()

    def forward(self, feats):
        return self.detect(list(feats))


class ViTBaseModel(nn.Module):
    def __init__(self, img_size=640, num_classes=1, use_p2=False, drop_path_rate=0.0):
        super().__init__()
        self.backbone = ViTDetBackbone(img_size=img_size, drop_path_rate=drop_path_rate)
        self.neck = SimpleFeaturePyramid(in_ch=self.backbone.embed_dim, out_ch=128, use_p2=use_p2)
        self.head = YoloHead(num_classes=num_classes, ch=(128, 128, 128), strides=self.neck.strides)

    def forward(self, x):
        return self.head(self.neck(self.backbone(x)))


vit_base = ViTBaseModel(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_p2=USE_P2,
                       drop_path_rate=DROP_PATH).to(DEVICE)
print(vit_base)


In [ ]:
# ViT-SAC model, closely matching the repo implementation style

class SAC(nn.Module):
    def __init__(self, dim, rates=(1, 3, 5)):
        super().__init__()
        self.rates = tuple(rates)
        self.branches = nn.ModuleList(
            nn.Conv2d(dim, dim, 3, padding=r, dilation=r, groups=dim) for r in self.rates
        )
        self.switch = nn.Conv2d(dim, len(self.rates), 1)
        nn.init.zeros_(self.switch.weight)
        nn.init.zeros_(self.switch.bias)
        self.gamma = nn.Parameter(torch.zeros(dim))
        self.register_buffer('gate_sum', torch.zeros(len(self.rates)), persistent=False)
        self.n = 0

    def forward(self, y, f, p):
        g = self.switch(p).float().softmax(1)
        out = sum(g[:, i:i + 1] * b(f) for i, b in enumerate(self.branches))
        if self.training:
            self.gate_sum += g.mean((0, 2, 3)).detach()
            self.n += 1
        return y + self.gamma * out.permute(0, 2, 3, 1)

    def pop_gates(self):
        g = self.gate_sum / max(self.n, 1)
        self.gate_sum.zero_()
        self.n = 0
        return g


class QKVSAC(nn.Module):
    def __init__(self, dim, rates):
        super().__init__()
        self.q, self.k, self.v = SAC(dim, rates), SAC(dim, rates), SAC(dim, rates)
        self.pool = nn.AvgPool2d(5, stride=1, padding=2)

    def forward(self, y):
        f = y.permute(0, 3, 1, 2)
        p = self.pool(f)
        return self.q(y, f, p), self.k(y, f, p), self.v(y, f, p)


class ViTDetSACBackbone(ViTDetBackbone):
    def __init__(self, *args, sac_rates=(1, 3, 5), **kw):
        super().__init__(*args, **kw)
        for blk in self.vit.blocks:
            blk.qkv_sac = QKVSAC(self.embed_dim, sac_rates)

    def forward(self, x):
        x = rearrange(self.vit.patch_embed.proj(x), 'b c h w -> b h w c')
        x = x + self.pos_embed
        B, H, W, C = x.shape
        ws = self.window_size

        for i, blk in enumerate(self.vit.blocks):
            y = blk.norm1(x)
            yq, yk, yv = blk.qkv_sac(y)

            q = blk.attn.qkv(y)
            q, k, v = q.chunk(3, dim=-1)
            q = q + yq
            k = k + yk
            v = v + yv

            q = rearrange(q, 'b n (h d) -> b h n d', h=blk.attn.num_heads)
            k = rearrange(k, 'b n (h d) -> b h n d', h=blk.attn.num_heads)
            v = rearrange(v, 'b n (h d) -> b h n d', h=blk.attn.num_heads)
            out = F.scaled_dot_product_attention(q, k, v)
            out = blk.attn.proj(rearrange(out, 'b h n d -> b n (h d)'))

            x = x + blk.drop_path1(blk.ls1(out))
            x = x + blk.drop_path2(blk.ls2(blk.mlp(blk.norm2(x))))

        x = self.vit.norm(x)
        return rearrange(x, 'b h w c -> b c h w')


class ViTSACModel(nn.Module):
    def __init__(self, img_size=640, num_classes=1, use_p2=False, drop_path_rate=0.0, sac_rates=(1, 3, 5)):
        super().__init__()
        self.backbone = ViTDetSACBackbone(img_size=img_size, drop_path_rate=drop_path_rate, sac_rates=sac_rates)
        self.neck = SimpleFeaturePyramid(in_ch=self.backbone.embed_dim, out_ch=128, use_p2=use_p2)
        self.head = YoloHead(num_classes=num_classes, ch=(128, 128, 128), strides=self.neck.strides)

    def forward(self, x):
        return self.head(self.neck(self.backbone(x)))


vit_sac = ViTSACModel(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_p2=USE_P2,
                     drop_path_rate=DROP_PATH, sac_rates=(1, 3, 5)).to(DEVICE)
print(vit_sac)


In [ ]:
# Dataset loader and training logic closely aligned with the repo

class YoloDataset(Dataset):
    def __init__(self, items, img_size=IMG_SIZE, augment=False, mosaic=1.0, mixup=0.0,
                 erase=0.0, photo_ops=0, photo_mag=0.4, zoom=(0.5, 1.3)):
        self.items = list(items)
        self.img_size = img_size
        self.augment = augment
        self.mosaic_p = mosaic
        self.mixup_p = mixup
        self.erase_p = erase
        self.photo_ops = photo_ops
        self.photo_mag = photo_mag
        self.zoom = zoom

        self.color = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.02)
        self.eraser = T.RandomErasing(p=1.0, scale=(0.02, 0.15), ratio=(0.3, 3.3), value='random')
        self.transform = T.Compose([T.ToTensor(), T.Normalize(mean=MEAN, std=STD)])

    def __len__(self):
        return len(self.items)

    @staticmethod
    def read_labels(label_path):
        if not label_path.exists():
            return np.zeros((0, 5), dtype=np.float32)
        rows = []
        for line in label_path.read_text().strip().splitlines():
            if not line.strip():
                continue
            parts = [float(x) for x in line.split()]
            if len(parts) == 5:
                rows.append(parts)
        return np.array(rows, dtype=np.float32).reshape(-1, 5) if rows else np.zeros((0, 5), dtype=np.float32)

    def _load(self, path):
        image = Image.open(path).convert('RGB')
        w0, h0 = image.size
        lab = self.read_labels(path.with_suffix('.txt').replace('/images/', '/labels/'))
        if len(lab):
            boxes = np.stack([
                (lab[:, 1] - lab[:, 3] / 2) * w0,
                (lab[:, 2] - lab[:, 4] / 2) * h0,
                (lab[:, 1] + lab[:, 3] / 2) * w0,
                (lab[:, 2] + lab[:, 4] / 2) * h0,
            ], 1).astype(np.float32)
        else:
            boxes = np.zeros((0, 4), dtype=np.float32)
        return image, lab[:, 0], boxes

    @staticmethod
    def _clip_filter(boxes, cls, S):
        if not len(boxes):
            return boxes, cls
        full = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        boxes = boxes.clip(0, S)
        bw, bh = boxes[:, 2] - boxes[:, 0], boxes[:, 3] - boxes[:, 1]
        keep = (bw > 2) & (bh > 2) & (bw * bh > 0.3 * full)
        return boxes[keep], cls[keep]

    def _single(self, idx):
        S = self.img_size
        image, cls, boxes = self._load(self.items[idx][0])
        w0, h0 = image.size
        scale = S / max(w0, h0)
        if self.augment:
            scale *= random.uniform(*self.zoom)
        new_w, new_h = max(1, round(w0 * scale)), max(1, round(h0 * scale))
        image = image.resize((new_w, new_h), Image.BILINEAR)
        if self.augment:
            pad_x = random.randint(min(0, S - new_w), max(0, S - new_w))
            pad_y = random.randint(min(0, S - new_h), max(0, S - new_h))
        else:
            pad_x, pad_y = (S - new_w) // 2, (S - new_h) // 2
        canvas = Image.new('RGB', (S, S), (114, 114, 114))
        canvas.paste(image, (pad_x, pad_y))
        if len(boxes):
            boxes = boxes * scale + np.array([pad_x, pad_y, pad_x, pad_y], dtype=np.float32)
        boxes, cls = self._clip_filter(boxes, cls, S)
        return canvas, cls, boxes

    def _mosaic(self, idx):
        S = self.img_size
        ids = [idx] + random.choices(range(len(self)), k=3)
        xc, yc = (int(random.uniform(0.5 * S, 1.5 * S)) for _ in range(2))
        canvas = Image.new('RGB', (2 * S, 2 * S), (114, 114, 114))
        all_boxes, all_cls = [], []
        for k, j in enumerate(ids):
            image, cls, boxes = self._load(self.items[j][0])
            w0, h0 = image.size
            r = S / max(w0, h0) * random.uniform(0.5, 1.0)
            w, h = max(1, round(w0 * r)), max(1, round(h0 * r))
            image = image.resize((w, h), Image.BILINEAR)
            ox, oy = [(xc - w, yc - h), (xc, yc - h), (xc - w, yc), (xc, yc)][k]
            canvas.paste(image, (ox, oy))
            if len(boxes):
                all_boxes.append(boxes * r + np.array([ox, oy, ox, oy], dtype=np.float32))
                all_cls.append(cls)
        off = S // 2
        if all_boxes:
            boxes, cls = np.concatenate(all_boxes) - off, np.concatenate(all_cls)
        else:
            boxes, cls = np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)
        boxes, cls = self._clip_filter(boxes, cls, S)
        return canvas.crop((off, off, off + S, off + S)), cls, boxes

    def _base(self, idx):
        if self.augment and random.random() < self.mosaic_p:
            return self._mosaic(idx)
        return self._single(idx)

    def __getitem__(self, idx):
        S = self.img_size
        canvas, cls, boxes = self._base(idx)

        if self.augment and random.random() < self.mixup_p:
            canvas2, cls2, boxes2 = self._base(random.randrange(len(self)))
            canvas = Image.blend(canvas, canvas2, float(np.random.beta(32.0, 32.0)))
            boxes, cls = np.concatenate([boxes, boxes2]), np.concatenate([cls, cls2])

        if self.augment:
            canvas = self.color(canvas)
            for op in random.sample([
                lambda im, m: T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)(im),
                lambda im, m: ImageOps.autocontrast(im),
                lambda im, m: ImageOps.equalize(im),
            ], self.photo_ops):
                canvas = op(canvas, self.photo_mag)
            if random.random() < 0.5:
                canvas = canvas.transpose(Image.FLIP_LEFT_RIGHT)
                if len(boxes):
                    boxes = np.stack([S - boxes[:, 2], boxes[:, 1], S - boxes[:, 0], boxes[:, 3]], 1)

        if len(boxes):
            target = np.stack([
                cls,
                (boxes[:, 0] + boxes[:, 2]) / 2 / S,
                (boxes[:, 1] + boxes[:, 3]) / 2 / S,
                (boxes[:, 2] - boxes[:, 0]) / S,
                (boxes[:, 3] - boxes[:, 1]) / S,
            ], 1).astype(np.float32)
        else:
            target = np.zeros((0, 5), dtype=np.float32)

        image = self.transform(canvas)
        if self.augment and random.random() < self.erase_p:
            image = self.eraser(image)
        return image, torch.from_numpy(target), str(self.items[idx][0])


def collate_fn(batch):
    images, targets, paths = zip(*batch)
    return torch.stack(images, 0), list(targets), list(paths)


def build_folder_loaders(root, splits=('train', 'val'), batch_size=8, img_size=IMG_SIZE,
                         augment=True, num_workers=4, aug=None):
    root = Path(root)
    loaders = []
    aug = aug or {}
    for split in splits:
        paths = sorted((root / 'images' / split).iterdir())
        assert paths, f'{root}/images/{split} is empty'
        ds = YoloDataset([(p, root / 'labels' / split / f'{p.stem}.txt') for p in paths if p.suffix.lower() in IMG_EXTS],
                         img_size=img_size, augment=(augment and split == 'train'), **aug)
        loaders.append(DataLoader(ds, batch_size=batch_size, shuffle=(split == 'train'),
                                  num_workers=num_workers, collate_fn=collate_fn))
    return loaders


def make_batch(targets, device):
    idx = torch.cat([torch.full((len(t),), i, dtype=torch.float32) for i, t in enumerate(targets)])
    t = torch.cat(targets)
    return {
        'batch_idx': idx.to(device),
        'cls': t[:, :1].to(device),
        'bboxes': t[:, 1:].to(device),
    }


class DetectionLoss:
    def __init__(self, model):
        self.criterion = v8DetectionLoss(model)

    def __call__(self, preds, targets, device):
        loss, loss_items = self.criterion(preds, make_batch(targets, device))
        if isinstance(loss_items, dict):
            vals = list(loss_items.values())
            loss_items = torch.stack([
                torch.as_tensor(v, dtype=torch.float32).detach().reshape(()).cpu() for v in vals
            ])
        else:
            loss_items = loss_items.detach().cpu()
        return loss.sum(), loss_items


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total, items = 0.0, torch.zeros(3)
    for images, targets, _ in loader:
        images = images.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            preds = model(images)
            loss, loss_items = criterion(preds, targets, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        total += loss.item()
        items += loss_items[:3].cpu()
    n = len(loader)
    return total / n, items / n


def evaluate_model(model, loader, device, img_size=IMG_SIZE, conf=0.001):
    model.eval()
    m = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    m25 = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', iou_thresholds=[0.25])
    m.warn_on_many_detections = m25.warn_on_many_detections = False
    t_tot, n_img = 0.0, 0
    for bi, (images, targets, _) in enumerate(loader):
        images = images.to(device)
        t0 = time.perf_counter()
        with torch.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            y = model(images)
        dt = time.perf_counter() - t0
        if bi >= 3:
            t_tot += dt; n_img += len(images)

        dets = y
        preds, gts = [], []
        for d, t in zip(dets, targets):
            if isinstance(d, tuple):
                d = d[0]
            if hasattr(d, 'shape') and d.dim() > 1:
                preds.append({'boxes': d[:, :4].cpu(), 'scores': d[:, 4].cpu(), 'labels': d[:, 5].long().cpu()})
            b, l = targets_to_xyxy(t, img_size)
            gts.append({'boxes': b, 'labels': l})
        m.update(preds, gts)
        m25.update(preds, gts)

    r, r25 = m.compute(), m25.compute()
    g = lambda x: float('nan') if x.item() < 0 else x.item()
    return {
        'mAP25': r25['map'].item(),
        'mAP50': r['map_50'].item(),
        'mAP': r['map'].item(),
        'mAP75': r['map_75'].item(),
        'mAP_small': g(r['map_small']),
        'mAP_medium': g(r['map_medium']),
        'mAP_large': g(r['map_large']),
        'AR100': r['mar_100'].item(),
        'ms_per_img': (t_tot / n_img * 1000) if n_img else float('nan'),
    }


def targets_to_xyxy(t, size):
    cx, cy, w, h = t[:, 1] * size, t[:, 2] * size, t[:, 3] * size, t[:, 4] * size
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1), t[:, 0].long()


# build the train/val loaders in repo style
aug_cfg = AUG_PRESETS['A']
train_loader = build_folder_loaders(FOCI_PREP, splits=('train', 'val'), batch_size=BATCH_SIZE,
                                    img_size=IMG_SIZE, augment=True, num_workers=2, aug=aug_cfg)[0]
val_loader = build_folder_loaders(FOCI_PREP, splits=('train', 'val'), batch_size=BATCH_SIZE,
                                  img_size=IMG_SIZE, augment=False, num_workers=2, aug=aug_cfg)[1]
print('Train loader length:', len(train_loader))
print('Val loader length:', len(val_loader))


In [ ]:
# Full experiment runner close to original repo
# Default: only aug A is used. You can extend AUG_OPTIONS to include B/C/D.


def run_model_experiment(model_name='vit_base', aug_name='A', seed=42, epochs=60, sac_rates=None):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    aug_cfg = AUG_PRESETS[aug_name]
    train_loader = build_folder_loaders(FOCI_PREP, splits=('train',), batch_size=BATCH_SIZE,
                                        img_size=IMG_SIZE, augment=True, num_workers=2, aug=aug_cfg)[0]
    val_loader = build_folder_loaders(FOCI_PREP, splits=('val',), batch_size=BATCH_SIZE,
                                      img_size=IMG_SIZE, augment=False, num_workers=2, aug=aug_cfg)[0]

    if model_name == 'vit_base':
        model = ViTBaseModel(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_p2=USE_P2,
                             drop_path_rate=DROP_PATH).to(DEVICE)
    else:
        model = ViTSACModel(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_p2=USE_P2,
                            drop_path_rate=DROP_PATH, sac_rates=sac_rates or (1, 3, 5)).to(DEVICE)

    criterion = DetectionLoss(model)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                 lr=LR_BACKBONE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_map = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        if epoch == epochs - CLOSE_MOSAIC + 1 and (aug_cfg['mosaic'] > 0 or aug_cfg['mixup'] > 0):
            train_loader.dataset.mosaic_p = 0.0
            train_loader.dataset.mixup_p = 0.0
            print(f'epoch {epoch}: mosaic + mixup off')

        tr_loss, it = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        scheduler.step()

        if epoch % EVAL_EVERY == 0 or epoch == 1 or epoch == epochs:
            val_metrics = evaluate_model(model, val_loader, DEVICE, IMG_SIZE, conf=0.01)
            history.append({'epoch': epoch, 'train_loss': float(tr_loss), 'val_map': float(val_metrics['mAP'])})
            print(f"epoch {epoch:3d} | loss {tr_loss:8.3f} | val mAP@0.5 {val_metrics['mAP50']:.4f} | mAP {val_metrics['mAP']:.4f}")

            if val_metrics['mAP'] > best_map:
                best_map = val_metrics['mAP']
                torch.save(model.state_dict(), CHECKPOINT_DIR / f'{model_name}_{aug_name}_s{seed}_best.pt')

        torch.save(model.state_dict(), CHECKPOINT_DIR / f'{model_name}_{aug_name}_s{seed}_last.pt')

    result = {
        'model': model_name,
        'aug': aug_name,
        'seed': seed,
        'epochs': epochs,
        'sac_rates': list(sac_rates) if sac_rates else None,
        'best_map': best_map,
        'history': history,
    }
    return model, result


# Unless you explicitly add more in AUG_OPTIONS, this notebook only runs AUG A.
# You can set AUG_OPTIONS = ['A', 'B'] or ['A', 'C', 'D'] if needed.
all_results = []
for model_name in ['vit_base', 'vit_sac']:
    for aug_name in AUG_OPTIONS:
        for seed in SEED_VALUES:
            sac_rates = (1, 3, 5) if model_name == 'vit_sac' else None
            _, result = run_model_experiment(model_name=model_name, aug_name=aug_name, seed=seed,
                                             epochs=EPOCHS, sac_rates=sac_rates)
            all_results.append(result)
            print('Result saved for:', model_name, aug_name, seed)

print('Completed experiment runs:', len(all_results))


In [ ]:
# Final evaluation report, close to original repo: per-config summary in markdown-like output


results = {}
for run in all_results:
    model_label = run['model']
    cfg_label = f"{run['aug']}|{run['seed']}"
    results.setdefault(model_label, {})[cfg_label] = run

summary_rows = []
for model_label, runs in results.items():
    for cfg_label, run in runs.items():
        summary_rows.append({
            'model': model_label,
            'aug': run['aug'],
            'seed': run['seed'],
            'best_map': run['best_map'],
            'epochs': run['epochs'],
            'sac_rates': run['sac_rates'],
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df)
summary_df.to_csv(RESULTS_DIR / 'foci_experiment_summary.csv', index=False)
print('Saved summary CSV to', RESULTS_DIR / 'foci_experiment_summary.csv')

# Simple mean summary by model and aug
mean_summary = summary_df.groupby(['model', 'aug'])['best_map'].mean().reset_index()
print(mean_summary)

mean_summary.to_csv(RESULTS_DIR / 'foci_mean_by_model_aug.csv', index=False)


In [ ]:
# Model comparison plot

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=mean_summary, x='aug', y='best_map', hue='model')
ax.set_title('Mean best validation mAP by augmentation preset')
ax.set_ylabel('best mAP')
ax.set_xlabel('augmentation')
plt.tight_layout()
plt.savefig(FIG_DIR / 'foci_aug_summary.png', dpi=150)
plt.show()


In [ ]:
# Save final outputs and training metadata
final_summary = {
    'seeds': SEED_VALUES,
    'epochs': EPOCHS,
    'augmentations': list(AUG_PRESETS.keys()),
    'results': all_results,
}
(RESULTS_DIR / 'foci_full_summary.json').write_text(json.dumps(final_summary, indent=2))
print('Saved final summary JSON to', RESULTS_DIR / 'foci_full_summary.json')


In [ ]:
# Final notebook entry point
print('Notebook is ready for repo-style ViTBase and ViT-SAC experiments on FOCI.')
print('Use the training cell to run the full sweep, or adjust aug / seed / epochs values before running.')


In [ ]:
# ---------------------------
# 12) Run the Full Notebook End-to-End
# ---------------------------

print('Notebook setup complete.')
print('This notebook consolidates the FOCI workflow for ViTBase and ViT-SAC.')
print('Next steps: adjust hyperparameters, run training cells, then review the saved metrics and visualizations.')
